In [2]:
!pip install --upgrade binance-historical-data plotly -q
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.8 MB/s eta 0:00:00


In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from binance_historical_data import BinanceDataDumper
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
warnings.filterwarnings('ignore')

In [4]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data


In [5]:
data_frequency = "1m"
start_date = "2020-05-01"
end_date = "2025-07-01"
tickers =['BTCUSDT']
download_ticker(tickers[0], start_date, end_date, data_frequency)
data = process_csv_files(tickers[0], data_frequency)
data

---> Found overall tickers: 596
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1m
---> Start Date: 20200501
---> End Date: 20250701


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/62 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/1 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 62 months 1 days
---> For BTCUSD new data saved for: 0 months 0 days


,open_time,Open,High,Low,Close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
399950,2020-05-01 00:01:00,8641.80,8670.00,8640.00,8652.21,191.526439,2020-05-01 00:01:59.999000,1.657609e+06,1446,102.905407,890475.606125,0
399951,2020-05-01 00:02:00,8653.27,8654.05,8624.94,8627.50,61.080967,2020-05-01 00:02:59.999000,5.276232e+05,660,30.623472,264462.526045,0
399952,2020-05-01 00:03:00,8627.47,8648.96,8625.29,8637.77,57.028320,2020-05-01 00:03:59.999000,4.926651e+05,573,23.950230,206880.995322,0
399953,2020-05-01 00:04:00,8638.38,8650.07,8630.00,8634.33,43.724565,2020-05-01 00:04:59.999000,3.778437e+05,527,16.374201,141516.822649,0
399954,2020-05-01 00:05:00,8634.33,8649.70,8634.12,8639.56,34.801463,2020-05-01 00:05:59.999000,3.007121e+05,387,16.559057,143072.499513,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1434,2025-07-01 23:55:00,105594.33,105664.93,105594.33,105664.92,5.337360,2025-07-01 23:55:59.999999,5.636827e+05,2560,4.108760,433935.088688,0
1435,2025-07-01 23:56:00,105664.93,105664.93,105642.41,105664.92,2.041990,2025-07-01 23:56:59.999999,2.157539e+05,1288,1.191010,125840.498243,0
1436,2025-07-01 23:57:00,105664.92,105681.14,105664.92,105681.14,2.671590,2025-07-01 23:57:59.999999,2.823283e+05,709,0.386610,40854.449322,0
1437,2025-07-01 23:58:00,105681.14,105702.39,105681.13,105681.14,6.274630,2025-07-01 23:58:59.999999,6.631517e+05,1357,2.083290,220172.209888,0


In [ ]:
import pandas as pd
import numpy as np

def backtest_ma_cross_1min(data, short_ma=50, long_ma=200, std_period=20, sl_mult=2, tp_mult=3, capital=10000, risk_per_trade=0.01, fee_percent=0.05):
    """
    Backtest MA crossover strategy on 1-minute timeframe with SL/TP based on std of short MA, including 0.05% fee per order.

    Parameters:
    - data: DataFrame with columns ['open_time', 'Open', 'High', 'Low', 'Close']
    - short_ma: Periods for short MA (in minutes)
    - long_ma: Periods for long MA (in minutes)
    - std_period: Periods for standard deviation of short MA
    - sl_mult: Multiplier for SL (std of short MA)
    - tp_mult: Multiplier for TP (std of short MA)
    - capital: Starting capital ($)
    - risk_per_trade: Risk per trade as fraction of capital
    - fee_percent: Fee per order as percentage of trade value (e.g., 0.05 for 0.05%)
    """
    # Ensure open_time is datetime
    data['open_time'] = pd.to_datetime(data['open_time'])

    # Calculate MAs
    data['SMA_short'] = data['Close'].rolling(window=short_ma).mean()
    data['SMA_long'] = data['Close'].rolling(window=long_ma).mean()

    # Calculate standard deviation of the short MA
    data['SMA_std'] = data['SMA_short'].rolling(window=std_period).std()

    # Initialize variables
    position = 0  # 0: no position, 1: long, -1: short
    equity = [capital]
    trades = []
    portfolio_value = capital

    for i in range(1, len(data)):
        # Skip rows where MAs or std are NaN
        if pd.isna(data['SMA_short'].iloc[i]) or pd.isna(data['SMA_long'].iloc[i]) or pd.isna(data['SMA_std'].iloc[i]):
            equity.append(portfolio_value)
            continue

        # Check for crossover signals
        if data['SMA_short'].iloc[i-1] < data['SMA_long'].iloc[i-1] and data['SMA_short'].iloc[i] > data['SMA_long'].iloc[i]:
            # Bullish crossover (long entry)
            if position == 0:
                entry_price = data['Close'].iloc[i]
                std = data['SMA_std'].iloc[i]
                sl = entry_price - sl_mult * std
                tp = entry_price + tp_mult * std
                risk = portfolio_value * risk_per_trade
                size = risk / (sl_mult * std) if std != 0 else 0  # Avoid division by zero
                if size > 0:  # Only enter if valid size
                    entry_value = entry_price * size
                    entry_fee = entry_value * (fee_percent / 100)
                    portfolio_value -= entry_fee  # Deduct entry fee
                    position = 1
                    trades.append({
                        'entry_time': data['open_time'].iloc[i],
                        'entry_price': entry_price,
                        'sl': sl,
                        'tp': tp,
                        'size': size,
                        'type': 'long',
                        'entry_fee': entry_fee
                    })

        elif data['SMA_short'].iloc[i-1] > data['SMA_long'].iloc[i-1] and data['SMA_short'].iloc[i] < data['SMA_long'].iloc[i]:
            # Bearish crossover (short entry)
            if position == 0:
                entry_price = data['Close'].iloc[i]
                std = data['SMA_std'].iloc[i]
                sl = entry_price + sl_mult * std
                tp = entry_price - tp_mult * std
                risk = portfolio_value * risk_per_trade
                size = risk / (sl_mult * std) if std != 0 else 0
                if size > 0:
                    entry_value = entry_price * size
                    entry_fee = entry_value * (fee_percent / 100)
                    portfolio_value -= entry_fee  # Deduct entry fee
                    position = -1
                    trades.append({
                        'entry_time': data['open_time'].iloc[i],
                        'entry_price': entry_price,
                        'sl': sl,
                        'tp': tp,
                        'size': size,
                        'type': 'short',
                        'entry_fee': entry_fee
                    })

        # Check SL/TP for open position
        if position != 0:
            trade = trades[-1]
            if position == 1:  # Long position
                if data['Low'].iloc[i] <= trade['sl']:
                    exit_price = trade['sl']
                    profit = (exit_price - trade['entry_price']) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee
                elif data['High'].iloc[i] >= trade['tp']:
                    exit_price = trade['tp']
                    profit = (exit_price - trade['entry_price']) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee
                # Check for opposite crossover
                elif data['SMA_short'].iloc[i] < data['SMA_long'].iloc[i]:
                    exit_price = data['Close'].iloc[i]
                    profit = (exit_price - trade['entry_price']) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee
            elif position == -1:  # Short position
                if data['High'].iloc[i] >= trade['sl']:
                    exit_price = trade['sl']
                    profit = (trade['entry_price'] - exit_price) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee
                elif data['Low'].iloc[i] <= trade['tp']:
                    exit_price = trade['tp']
                    profit = (trade['entry_price'] - exit_price) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee
                # Check for opposite crossover
                elif data['SMA_short'].iloc[i] > data['SMA_long'].iloc[i]:
                    exit_price = data['Close'].iloc[i]
                    profit = (trade['entry_price'] - exit_price) * trade['size']
                    exit_value = exit_price * trade['size']
                    exit_fee = exit_value * (fee_percent / 100)
                    portfolio_value += profit - exit_fee  # Deduct exit fee
                    position = 0
                    trade['exit_time'] = data['open_time'].iloc[i]
                    trade['exit_price'] = exit_price
                    trade['profit'] = profit
                    trade['exit_fee'] = exit_fee
                    trade['total_fees'] = trade['entry_fee'] + exit_fee

        equity.append(portfolio_value)

    # Calculate performance metrics
    equity = pd.Series(equity)
    total_return = (equity.iloc[-1] / equity.iloc[0] - 1) * 100

    # Annualized return for 1-minute timeframe (6.5-hour trading day, 252 days/year)
    time_delta = (data['open_time'].iloc[-1] - data['open_time'].iloc[0]).total_seconds() / (60 * 60 * 24)
    bars_per_day = 6.5 * 60  # 390 minutes per trading day
    trading_days = time_delta / (6.5 / 24)
    n_years = trading_days / 252
    annualized_return = ((equity.iloc[-1] / equity.iloc[0]) ** (1 / n_years) - 1) * 100 if n_years > 0 else 0

    # Sharpe ratio adjusted for 1-minute timeframe
    returns = equity.pct_change().dropna()
    bars_per_year = 252 * bars_per_day  # ~98,280 1-minute bars per year
    sharpe_ratio = (returns.mean() * np.sqrt(bars_per_year) - 0.02) / (returns.std() * np.sqrt(bars_per_year)) if returns.std() != 0 else 0

    drawdowns = (equity.cummax() - equity) / equity.cummax()
    max_drawdown = drawdowns.max() * 100
    win_trades = sum(1 for t in trades if t.get('profit', 0) > t.get('total_fees', 0))
    win_rate = (win_trades / len(trades) * 100) if trades else 0
    total_fees = sum(t.get('total_fees', 0) for t in trades)

    # Output results
    print(f"Total Return: {total_return:.2f}%")
    print(f"Annualized Return: {annualized_return:.2f}%")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Max Drawdown: {max_drawdown:.2f}%")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Number of Trades: {len(trades)}")
    print(f"Total Fees Paid: ${total_fees:.2f}")

    return trades, equity

# Example usage (uncomment and adjust with your data)
# data = pd.read_csv('your_ohlc_data.csv')
# data['open_time'] = pd.to_datetime(data['open_time'])
trades, equity = backtest_ma_cross_1min(data, short_ma=50, long_ma=20, std_period=50, sl_mult=2, tp_mult=5, fee_percent=0.05)